# 04 — Petrobras 3W contract challenge


Challenge the supposedly neutral contract with a real, non-telecom source.
The adapter chooses the smallest deterministic subset that covers normal,
transient, persistent, transition, and missing/frozen cases, then records
what fits, what cannot be expressed, and what must not be invented.


    This is a **thin orchestration notebook**. The tested implementation lives
    in the GitHub package; this notebook only sets paths, calls one workflow,
    and displays its evidence. Run cells from top to bottom.

In [ ]:
# Shared implementation: GitHub in Colab, local source when testing this repository.
import os
import subprocess
import sys
from pathlib import Path

REPOSITORY = "https://github.com/LiliDopidze/anomaly_detection.git"
RUNTIME_REF = os.getenv("ANOMALY_RUNTIME_REF", "main")
LOCAL_SOURCE = os.getenv("ANOMALY_SOURCE_ROOT")

if LOCAL_SOURCE:
    sys.path.insert(0, str(Path(LOCAL_SOURCE).resolve()))
    RUNTIME_COMMIT = "local-working-tree"
else:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "--quiet",
        f"git+{REPOSITORY}@{RUNTIME_REF}",
    ])
    RUNTIME_COMMIT = subprocess.check_output(
        ["git", "ls-remote", REPOSITORY, RUNTIME_REF], text=True
    ).split()[0]

os.environ["ANOMALY_RUNTIME_REF"] = RUNTIME_REF
os.environ["ANOMALY_RUNTIME_COMMIT"] = RUNTIME_COMMIT
print(f"Runtime: {RUNTIME_REF} ({RUNTIME_COMMIT[:12]})")

In [ ]:
# Mount Google Drive in Colab. Local validation skips this block.
if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")

DRIVE_ROOT = Path(os.getenv(
    "ANOMALY_DRIVE_ROOT",
    "/content/drive/MyDrive/anomaly_detection",
))
print(f"Drive root: {DRIVE_ROOT}")

## Configuration

`THREEW_SOURCE` must point at the folder containing `dataset.ini` and
directories `0` through `9`. The selected public fixture is copied to a
separate immutable fixture folder for repeatable tests.

In [ ]:
THREEW_SOURCE = Path(os.getenv(
    "ANOMALY_THREEW_SOURCE",
    str(
        DRIVE_ROOT
        / "sources"
        / "petrobras_3w"
        / "2.0.0"
        / "raw"
        / "3w_dataset_2.0.0"
    ),
))
OUTPUT_ROOT = Path(os.getenv(
    "ANOMALY_OUTPUT_ROOT",
    str(DRIVE_ROOT / "outputs" / "milestone_1" / "v0.3" / "petrobras_3w"),
))
RUN_ID = os.getenv("ANOMALY_RUN_ID", "threew_contract_v1")
RUN_ROOT = OUTPUT_ROOT / RUN_ID
FIXTURE_DESTINATION = Path(os.getenv(
    "ANOMALY_FIXTURE_DESTINATION",
    str(
        DRIVE_ROOT
        / "fixtures"
        / "petrobras_3w"
        / "2.0.0"
        / "v0.3"
        / RUN_ID
    ),
))

print(f"Source: {THREEW_SOURCE}")
print(f"New immutable run: {RUN_ROOT}")
print(f"New immutable fixture: {FIXTURE_DESTINATION}")

In [ ]:
from anomaly_detection.workflows import challenge_threew

challenge = challenge_threew(
    THREEW_SOURCE,
    RUN_ROOT,
    fixture_destination=FIXTURE_DESTINATION,
)

In [ ]:
from pprint import pprint

pprint({
    "all_criteria_covered":
        challenge["all_required_criteria_covered"],
    "selected_instances": challenge["selected"],
    "row_counts": challenge["row_counts"],
    "generic_contract_changes":
        challenge["contract_fit_report"][
            "generic_contract_changes_required"
        ],
    "not_representable_without_invention":
        challenge["contract_fit_report"][
            "not_representable_without_invention"
        ],
})
print(f"Evidence: {RUN_ROOT / 'workflow_report.json'}")